# QSV2G Stage 05 — FULL Kaggle evaluation (E1–E5)

**Default: `SMOKE=False` (full campaign)**. The local smoke campaign already completed in a separate run.
This notebook performs E1–E4 in software, then attempts official Tamarin + Maude installation and E5 symbolic proof checks.

**No physical quantum network or charger**; the formal model *assumes* the ideal QSS gate and only addresses an abstract A/B classical consent flow. A failed installation or incomplete proof is recorded as UNRESOLVED, not as successful formal verification.

**Planned workload, not previously measured:** 18,000 E1 transactions, 36 scripted E2 scenarios and 1,000 requested Hypothesis sequences, 300,000 E3 full-vector trials and 6,000 genuine-PQC endpoint-token trials across all four L values, 11 E4 controlled fault scenarios, and an E5 model proof attempt if binaries run. A Kaggle runtime limit may interrupt the run.

**Research reproducibility:** exact Git SHA pinned; fresh output directory, no overwrite of Stage 01–04; no publication claims before output audit.

## 00. Settings and pinned Git revision

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, zipfile, shutil
SMOKE = False  # FULL Stage 05 evaluation (the smoke run was audited separately)
PINNED_COMMIT = "b463fc758fa28f1853fe06c4130ee2f25173ac0c"  # Replaced after verified GitHub code commit
from datetime import datetime, timezone
ROOT=Path("/kaggle/working/QSV2G") if Path("/kaggle/working").exists() else Path.cwd()/"QSV2G_kaggle"
OUT_BASE=Path("/kaggle/working/qsv2g_stage05_results") if Path("/kaggle/working").exists() else Path.cwd()/"qsv2g_stage05_results"
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT=OUT_BASE/("smoke" if SMOKE else "final")/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
print("Fresh output directory:",OUT)
assert len(PINNED_COMMIT)==40 and all(x in "0123456789abcdef" for x in PINNED_COMMIT), "Set PINNED_COMMIT to the reported 40-character immutable Stage 05 source commit"
TOOLS_DIR=Path("/kaggle/working/qsv2g_stage05_e5_tools") if Path("/kaggle/working").exists() else Path.cwd()/"qsv2g_stage05_e5_tools"


def checkpoint_zip(stage):
    """Preserve completed outputs even if a later long Kaggle cell fails."""
    import hashlib
    artifacts=[]
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and path.name not in ("snapshot_manifest.json", "artifact_manifest.json"):
            artifacts.append({"path":str(path.relative_to(OUT)),"bytes":path.stat().st_size,
                              "sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
    (OUT/"snapshot_manifest.json").write_text(json.dumps({"git_sha":head,"smoke":SMOKE,
        "completed_through":stage,"files":artifacts},indent=2))
    zip_path=OUT.parent/(f"qsv2g_stage05_{'smoke' if SMOKE else 'final'}_{RUN_ID}_checkpoint.zip")
    temporary=zip_path.with_suffix('.tmp')
    with zipfile.ZipFile(temporary,"w",zipfile.ZIP_DEFLATED) as z:
        for path in sorted(OUT.rglob("*")):
            if path.is_file():z.write(path,path.relative_to(OUT.parent))
    temporary.replace(zip_path)
    print("Completed-stage checkpoint:",stage,zip_path)


## 01. Clone the approved source commit and record environment

In [ ]:
if not (ROOT/".git").exists():
    subprocess.run(["git","clone","--quiet","https://github.com/shanmukhdatta/QSV2G.git",str(ROOT)],check=True)
subprocess.run(["git","-C",str(ROOT),"fetch","--quiet","origin",PINNED_COMMIT],check=True)
subprocess.run(["git","-C",str(ROOT),"checkout","--quiet","--detach",PINNED_COMMIT],check=True)
head=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
assert head==PINNED_COMMIT
print("Source pinned:",head)

## 02. Dependencies and real-cryptography preflight

In [ ]:
import importlib.util
missing=[x for x in ["numpy","pandas","matplotlib","cryptography","pytest","hypothesis"] if importlib.util.find_spec(x) is None]
if missing:
    subprocess.run([sys.executable,"-m","pip","install", "--quiet", *missing],check=True)
sys.path.insert(0,str(ROOT/"src"))
from qsv2g_core import PQCrypto
try:
    backend=PQCrypto()
except RuntimeError:
    # Kaggle's system OpenSSL may be older; explicitly install real native PQC.
    subprocess.run([sys.executable,"-m","pip","install","--quiet","pqcrypto>=0.4"],check=True)
    backend=PQCrypto()  # Must still succeed with genuine ML-KEM/ML-DSA
print("Real cryptographic backend:",backend.kind)
(OUT/"git_revision.txt").write_text(head+"\n")
(OUT/"pip_freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))

## 03. Regression gate — must pass before the full campaign

In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q","tests"],cwd=ROOT,check=True)

## 04. E1 — matched authorization and controlled software cost

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e1","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)
checkpoint_zip("E1")


## 05. E2 — real-PQC adversarial tests and generated stateful sequences

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e2","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)
subprocess.run([sys.executable,"experiments/stage05/stateful_hypothesis.py",
    "--examples",str(20 if SMOKE else 1000),"--steps",str(15 if SMOKE else 50),
    "--output",str(OUT/"e2"/"hypothesis.json")],cwd=ROOT,check=True)
checkpoint_zip("E2")


## 06. E3 — all four vector lengths, 300k modeled vectors + 6k real-PQC EV token checks

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e3","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)
checkpoint_zip("E3")


## 07. E4 — authority outages and contention

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e4","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)
checkpoint_zip("E4")


## 08. E5 — optional official-binary installation, symbolic proof, and honest status reporting

In [ ]:
from pathlib import Path
E5_OUT=OUT/"e5"
E5_OUT.mkdir(parents=True,exist_ok=True)
# Releases are integrity-pinned by SHA-256; no apt/sudo or unknown pip package.
installer=subprocess.run([sys.executable,"experiments/stage05/install_tamarin.py",
    "--tools-dir",str(TOOLS_DIR),"--status",str(E5_OUT/"install_status.json")],
    cwd=ROOT,capture_output=True,text=True)
(E5_OUT/"e5_install_log.txt").write_text(installer.stdout+"\n"+installer.stderr)
print("Tamarin installer result:",json.loads((E5_OUT/"install_status.json").read_text()).get("status"))
proof=subprocess.run([sys.executable,"experiments/stage05/run_tamarin.py",
    "--out",str(E5_OUT),"--tools-dir",str(TOOLS_DIR),
    "--timeout",str(180 if SMOKE else 900)],cwd=ROOT,capture_output=True,text=True)
(E5_OUT/"e5_runner_log.txt").write_text(proof.stdout+"\n"+proof.stderr)
if proof.returncode:
    raise RuntimeError("E5 proof wrapper crashed, inspect e5_runner_log.txt")
print((E5_OUT/"e5_status.json").read_text())

checkpoint_zip("E5")


## 09. Statistical summaries and figures (explicit E3 per-length Tier-B token evidence)

In [ ]:
subprocess.run([sys.executable,"experiments/stage05/analyze.py",str(OUT)],cwd=ROOT,check=True)
print((OUT/"analysis_summary.json").read_text())

## 10. Archive raw E1–E4 data and E5 installation/proof evidence

In [ ]:
import hashlib
files=[]
for path in sorted(OUT.rglob("*")):
    if path.is_file() and path.name!="artifact_manifest.json":
        files.append({"path":str(path.relative_to(OUT)),"bytes":path.stat().st_size,"sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
(OUT/"artifact_manifest.json").write_text(json.dumps({"git_sha":head,"smoke":SMOKE,"files":files},indent=2))
archive=OUT.parent/(f"qsv2g_stage05_{'smoke' if SMOKE else 'final'}_{RUN_ID}.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as zipf:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():zipf.write(path,path.relative_to(OUT.parent))
print("Results archive:",archive)